# D1: DINOv2's depth setup on my 795 NYUv2 images (frozen ViT-S/14 + DPT, depth only)

DINOv2 reports NYUv2 RMSE 0.356 with this setup, trained on ≈ 24k NYU frames (Tab. 11). This run trains the same setup on my 795 labelled images, to test whether the gap between my models and 0.356 is the training data.

Only what DINOv2's setup prescribes is changed against my pipeline:
- backbone **frozen** (variant `frozen`, as my run A);
- **depth only** (`w_seg = 0`);
- same DPT decoder as DINOv2's for ViT-S (`dinov2/hub/depthers.py`: 256 channels, post-process channels 48/96/192/384, readout "project");
- regression with the SiLog loss.

Everything else is my NYU recipe (E2: no critic, 20k iterations, batch 8, augmentation as in notebook 02, seed 42, standard position embeddings), so **training data and recipe details** are the only differences from DINOv2's number.


## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
from mtl_train import *   # mtl_common + the training code of notebook 02 (critic, losses, optimizer, train_run)
from repro_common import PAPER   # the papers' reference numbers, defined once (checked against the PDFs)

configure()
print(f"Device: {device}, GPU: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'}")


Device: cuda, GPU: NVIDIA RTX A4000


## Training

In [3]:
D1 = dict(use_critic=False, w_seg=0.0, seed=42, ckpt_root=ROOT / "ckpt" / "d1", results_root=ROOT / "results" / "d1")
configure(**D1)
assert CFG["variants"]["frozen"]["n_trainable_blocks"] == 0
hist_d1 = train_run("nyu", "frozen")


nyu_frozen: already trained


## Evaluation
Full NYUv2 test set, Eigen crop, single pass (plus flip and sliding window as supplementary rows), compared with DINOv2 Tab. 11, DPT Tab. 2, SwinMTL Tab. III and my runs.

In [4]:
configure(**D1)
out_path = CFG["results_root"] / "eval" / "nyu_frozen.json"
if out_path.exists():
    R_D1 = load_json(out_path)
else:
    model = load_trained("nyu", "frozen")
    R_D1 = eval_nyu(model, slide=True)
    save_json(R_D1, out_path)
    del model
    torch.cuda.empty_cache()
configure()

def row(label, R, key="nyu_depth"):
    d = R[key]
    return {"model": label, "RMSE": d["rmse"], "AbsRel": d["abs_rel"], "δ1": d["d1"], "log10": d.get("log10")}

rows = [
    {"model": "DINOv2 ViT-S/14 + DPT, frozen, ≈ 24k frames (Tab. 11)", "RMSE": PAPER["dinov2_tab11_vits14"]["nyu"][2]},
    {"model": "DPT-Hybrid, fine-tuned (DPT Tab. 2)", "RMSE": PAPER["dpt_tab2"]["DPT-Hybrid"]["rmse"], "AbsRel": PAPER["dpt_tab2"]["DPT-Hybrid"]["abs_rel"], "δ1": PAPER["dpt_tab2"]["DPT-Hybrid"]["d1"], "log10": PAPER["dpt_tab2"]["DPT-Hybrid"]["log10"]},
    {"model": "SwinMTL (Tab. III)", "RMSE": PAPER["swinmtl_tab3"]["SwinMTL"][0]},
    row("D1: DINOv2 setup (frozen + DPT, depth only), 795 images", R_D1),
    row("D1 +flip (TTA)", R_D1, "nyu_depth_flip"),
    row("D1 +slide (TTA)", R_D1, "nyu_depth_slide"),
]
for label, root, name in [("A: frozen, multi-task, critic (notebook 02)", "results", "nyu_frozen"),
                          ("B: 4 blocks, multi-task, critic", "results", "nyu_partial_ft"),
                          ("E2: 4 blocks, multi-task, no critic", "results/nocritic", "nyu_partial_ft")]:
    R = load_json(ROOT / root / "eval" / f"{name}.json")
    if R:
        rows.append(row(label, R))
d1_table = pd.DataFrame(rows).set_index("model")
d1_table.to_csv(ROOT / "results" / "compare" / "ablation_d1_dinov2_setup_nyu.csv")
display(d1_table.style.format(lambda v: "—" if pd.isna(v) else f"{v:.4f}"))


,RMSE,AbsRel,δ1,log10
model,,,,
"DINOv2 ViT-S/14 + DPT, frozen, ≈ 24k frames (Tab. 11)",0.3560,—,—,—
"DPT-Hybrid, fine-tuned (DPT Tab. 2)",0.3570,0.1100,0.9040,0.0450
SwinMTL (Tab. III),0.5179,—,—,—
"D1: DINOv2 setup (frozen + DPT, depth only), 795 images",0.4199,0.1227,0.8629,0.0511
D1 +flip (TTA),0.4146,0.1212,0.8665,0.0505
D1 +slide (TTA),0.4144,0.1202,0.8675,0.0503
"A: frozen, multi-task, critic (notebook 02)",0.4671,0.1373,0.8323,0.0566
"B: 4 blocks, multi-task, critic",0.4329,0.1247,0.8570,0.0521
"E2: 4 blocks, multi-task, no critic",0.4282,0.1223,0.8601,0.0515
